Imports all libraries needed for the analysis and reads the cleaned survey CSV. The two-row Qualtrics header requires `header=[0,1]`; the first data row (containing question text) is then dropped to leave only responses.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix

survey_data = pd.read_csv("Cleaned_BookstoreSurvey.csv", header=[0, 1])
survey_data = survey_data.drop(survey_data.index[0]).reset_index(drop=True)
print(survey_data.shape)
print(survey_data.head(3))

(48, 26)
                                   Q1  \
  What is your affiliation with CWRU?   
0               Undergraduate student   
1               Undergraduate student   
2               Undergraduate student   

                                                            Q4  \
  Have you ever browsed the CWRU bookstore (in-person/online)?   
0                                                yes             
1                                                yes             
2                                                yes             

                                                                   Q6  \
  Have you ever purchased from the CWRU bookstore (in-person/online)?   
0                                                yes                    
1                                                yes                    
2                                                yes                    

                                                                 Q7  \
  What type of cl

***Feature Engineering — Encoding Ordinal & Categorical Variables***

Converts each survey question's text responses into numbers. Likert-scale items become ordinal scores (1–4 for likelihood, 1–6 for spending), yes/no questions become binary flags, and year-in-school is ranked 1–4. Rows with any missing value are dropped, leaving a fully numeric modeling dataset.

In [ ]:
q26_col = ("Q26", "How likely are you to purchase clothing from the CWRU bookstore next semester?")
q18_col = ("Q18", "How likely are you to purchase apparel after seeing an online advertisement?")
q21_col = ("Q21", "How likely are you to purchase bookstore clothing after seeing an in-person advertisement?")
q6_col  = ("Q6",  "Have you ever purchased from the CWRU bookstore (in-person/online)?")
q4_col  = ("Q4",  "Have you ever browsed the CWRU bookstore (in-person/online)?")
q8_col  = ("Q8",  "What is the most important factor you consider when deciding to purchase new apparel?")
q23_col = ("Q23", "In the last 3 months, what is the most you have spent on a single non-essential clothing item?")
q24_col = ("Q24", "On average how much do you spend on a single item of clothing during a typical shopping trip/purchase?")
q2_col  = ("Q2",  "What is your current year in school?")
q3_col  = ("Q3",  "What is your gender?")

likelihood_map = {
    "1- Very unlikely": 1, "1-  Very unlikely": 1,
    "2- Unlikely": 2,      "2-  Unlikely": 2,
    "3- Likely": 3,        "3-  Likely": 3,
    "4- Very likely": 4,   "4-  Very likely": 4,
}

spending_map = {"<$10": 1, "$10 - $20": 2, "$20 - $30": 3, "$30 - $40": 4, "$40 - $50": 5, "$50+": 6}
year_map     = {"Freshman": 1, "Sophomore": 2, "Junior": 3, "Senior": 4}

model_df = pd.DataFrame()
model_df["purchase_likelihood"]    = survey_data[q26_col].map(likelihood_map)
model_df["online_ad_likelihood"]   = survey_data[q18_col].map(likelihood_map)
model_df["inperson_ad_likelihood"] = survey_data[q21_col].map(likelihood_map)
model_df["has_purchased"]          = (survey_data[q6_col].str.strip().str.lower() == "yes").astype(int)
model_df["has_browsed"]            = (survey_data[q4_col].str.strip().str.lower() == "yes").astype(int)
model_df["price_priority"]         = (survey_data[q8_col].str.strip() == "Price").astype(int)
model_df["max_spend_score"]        = survey_data[q23_col].str.strip().map(spending_map)
model_df["avg_spend_score"]        = survey_data[q24_col].str.strip().map(spending_map)
model_df["year_in_school"]         = survey_data[q2_col].str.strip().map(year_map)
model_df["is_female"]              = (survey_data[q3_col].str.strip() == "Female").astype(int)

model_df = model_df.dropna().reset_index(drop=True)
print(f"Modeling dataset: {model_df.shape[0]} rows, {model_df.shape[1]} columns")
print(model_df.describe())

***Model 1 — Logistic Regression (Will the Student Purchase?)***

Creates a binary outcome — 1 if a student rated their purchase likelihood as 3 or 4, 0 otherwise — and fits a default logistic regression as a baseline. The classification report shows how well the model separates likely from unlikely buyers before any tuning is applied.

In [ ]:
model_df["will_purchase"] = (model_df["purchase_likelihood"] >= 3).astype(int)
print("Class distribution:\n", model_df["will_purchase"].value_counts())

features = ["online_ad_likelihood", "inperson_ad_likelihood", "has_purchased",
            "has_browsed", "price_priority", "max_spend_score", "avg_spend_score",
            "year_in_school", "is_female"]

X = model_df[features]
y = model_df["will_purchase"]

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X, y)

print(f"\nBaseline accuracy: {(log_reg.predict(X) == y).mean():.3f}")
print(classification_report(y, log_reg.predict(X), target_names=["Unlikely (0)", "Likely (1)"]))

Runs GridSearchCV across regularization strength (C) and solver algorithm using 5-fold stratified cross-validation, which keeps the same class balance in each fold. The best combination is refitted on the full dataset; its coefficients and odds ratios reveal which features most strongly predict purchase intent.

In [ ]:
param_grid = {
    "C":      [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs", "saga"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=cv,
    scoring="accuracy",
    refit=True
)
grid_search.fit(X, y)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}\n")

best_log_reg = grid_search.best_estimator_
y_pred = best_log_reg.predict(X)
print(classification_report(y, y_pred, target_names=["Unlikely (0)", "Likely (1)"]))

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": best_log_reg.coef_[0],
    "Odds Ratio": np.exp(best_log_reg.coef_[0])
}).sort_values("Coefficient", ascending=False)
print(coef_df.to_string(index=False))

Displays the tuned model's confusion matrix to show correct and incorrect classifications, alongside a coefficient bar chart where bar direction and length indicate how strongly each variable pushes a student toward or away from purchasing.

In [ ]:
cm = confusion_matrix(y, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Unlikely", "Likely"],
            yticklabels=["Unlikely", "Likely"])
axes[0].set_title("Logistic Regression — Confusion Matrix (Tuned)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

coef_df_sorted = coef_df.sort_values("Coefficient")
colors = ["tomato" if c < 0 else "steelblue" for c in coef_df_sorted["Coefficient"]]
axes[1].barh(coef_df_sorted["Feature"], coef_df_sorted["Coefficient"], color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Logistic Regression — Feature Coefficients (Tuned)")
axes[1].set_xlabel("Coefficient Value")

plt.tight_layout()
plt.show()

***Model 2 — K-Means Clustering (Student Segmentation)***

Determines the optimal number of student segments by running K-Means for k = 2 through 7, recording two metrics: inertia (lower means tighter clusters) via the elbow method, and silhouette score (higher means better-separated clusters). The k with the highest silhouette score is selected as the best.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cluster_features = ["purchase_likelihood", "online_ad_likelihood", "inperson_ad_likelihood",
                    "max_spend_score", "avg_spend_score"]
X_clust = model_df[cluster_features]

inertias = []
silhouette_scores = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_clust, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_range, inertias, marker="o", color="steelblue")
axes[0].set_title("Elbow Method — Inertia by k")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].set_xticks(list(k_range))

axes[1].plot(k_range, silhouette_scores, marker="o", color="coral")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score (higher = better)")
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f"Best k by silhouette score: {best_k}  (score = {max(silhouette_scores):.3f})")

Fits the final K-Means model with the optimal k, assigns each respondent to a segment, and prints the mean feature profile per cluster. Segments are labeled Low, Moderate, or High Interest based on their average purchase likelihood score.

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
model_df["cluster"] = kmeans.fit_predict(X_clust)

cluster_profile = model_df.groupby("cluster")[cluster_features + ["has_purchased"]].mean().round(2)
cluster_profile.index = [f"Segment {i}" for i in cluster_profile.index]
print("Cluster Profiles (mean values per segment):")
print(cluster_profile.to_string())

rank_labels = ["Low Interest", "Moderate Interest", "High Interest", "Very High Interest"]
sorted_segs = cluster_profile["purchase_likelihood"].sort_values().index.tolist()
label_map = {seg: rank_labels[i] for i, seg in enumerate(sorted_segs)}
cluster_profile["Label"] = cluster_profile.index.map(label_map)
print("\nSegment Labels:")
print(cluster_profile["Label"])

Plots each student as a point positioned by their online ad responsiveness and purchase likelihood, colored by segment. A separate bar chart shows the relative size of each group, giving a sense of how many students fall into each customer segment.

In [ ]:
palette = ["steelblue", "coral", "seagreen", "mediumpurple", "goldenrod"]
colors_map = {i: palette[i] for i in range(best_k)}
segment_labels = cluster_profile["Label"].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for seg, grp in model_df.groupby("cluster"):
    label = segment_labels.get(f"Segment {seg}", f"Segment {seg}")
    axes[0].scatter(grp["online_ad_likelihood"], grp["purchase_likelihood"],
                    label=label, color=colors_map[seg], alpha=0.8, edgecolors="white", s=70)
axes[0].set_title("Segments — Ad Responsiveness vs Purchase Likelihood")
axes[0].set_xlabel("Online Ad Likelihood (1–4)")
axes[0].set_ylabel("Purchase Likelihood (1–4)")
axes[0].legend()
axes[0].set_xticks([1, 2, 3, 4])
axes[0].set_yticks([1, 2, 3, 4])

cluster_sizes = model_df["cluster"].value_counts().sort_index()
bar_labels = [segment_labels.get(f"Segment {i}", f"Segment {i}") for i in cluster_sizes.index]
axes[1].bar(bar_labels, cluster_sizes.values, color=[colors_map[i] for i in cluster_sizes.index])
axes[1].set_title("Segment Sizes")
axes[1].set_xlabel("Segment")
axes[1].set_ylabel("Number of Respondents")

plt.tight_layout()
plt.show()